# Statistical Safety Validation of ML-Based Perception and Automated Driving

**Module 06 · AI Safety (ISO/PAS 8800)** · Notebook 7 of 7

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/06_AI_Safety_ISO_PAS_8800/07_statistical_safety_validation.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Intermediate | ~75 min | [04 Calibration](04_calibration.ipynb); [SOTIF simulation-based validation](../04_SOTIF_ISO21448/04_simulation_based_validation.ipynb); basic statistics | ISO 21448:2022, ISO/PAS 8800:2024, ISO 34502:2022, ISO/TS 5083:2025, UL 4600 (3rd ed., 2023) |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

- Explain why "drive until it is safe" does not work, and reproduce the **RAND "Driving to Safety"** numbers exactly.
- Distinguish **functional, logical and concrete scenarios** (Menzel et al., 2018) from the **PEGASUS 6-layer model** of scenario content (Scholtes et al., 2021).
- Measure **parameter-space coverage** correctly, and explain why coverage is not residual-risk evidence.
- Compute **zero-failure demonstration bounds**, **exact Poisson confidence intervals** and the miles needed for a **superiority test with power**.
- Choose safety-relevant perception metrics and turn acceptance criteria into statistically meaningful statements.
- Describe **safety performance indicators (SPIs)** and **shadow mode** as operation-phase evidence.

## Introduction

For conventionally specified software we verify against requirements and use coverage criteria. For ML-based perception:
- the input space (weather, light, objects, poses, occlusion, sensor states) cannot be enumerated;
- the "specification" is partly implicit in data;
- performance is a statistical property per ODD slice;
- rare corner cases dominate the residual risk.

**Kalra & Paddock (2016), RAND RR-1478 "Driving to Safety"**, using the 2013 US rate of **1.09 fatalities per 100 million miles** and a fleet of 100 vehicles driving 24 h a day, 365 days a year, at 25 mph:
- **275 million failure-free miles** are needed to show with 95 % confidence that the AV fatality rate is no higher than the human rate, about **12.5 years**;
- **8.8 billion miles** are needed to estimate the fatality rate to within ±20 % (95 % CI), about **400 years**;
- about **5 billion miles** (≈225 years) to show with 95 % confidence that the rate is lower, if the AV is truly 20 % better;
- **11 billion miles** (11.3 bn) to show with 95 % confidence **and 80 % power** that AVs are 20 % better, about **518 years**.

Hence validation combines several kinds of evidence: scenario-based testing, simulation, proving-ground tests, statistical field data, and a structured safety argument (ISO 21448, ISO/PAS 8800, ISO/TS 5083, UL 4600).

In [ ]:
from dataclasses import dataclass
from typing import List, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
OI = ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7", "#000000"]
pd.set_option("display.width", 160)

## 1. Scenario-based testing

### 1.1 Scenario abstraction levels (Menzel, Bagschik & Maurer, 2018)

| Level | Description | Example |
|---|---|---|
| **Functional scenario** | Linguistic, expert-readable description | "A pedestrian crosses a two-lane urban road in front of the ego vehicle" |
| **Logical scenario** | Parameter space: parameters with ranges or distributions | Pedestrian speed 0.5–2.0 m/s, distance 5–30 m, ego speed 20–50 km/h, weather ∈ {clear, rain, fog} |
| **Concrete scenario** | One value per parameter; directly executable | Speed 1.3 m/s, distance 12 m, ego 38 km/h, rain, dusk |

The **ODD** (taxonomy in ISO 34503:2023) bounds the parameter ranges of the logical scenarios. ISO 34502:2022 describes a scenario-based safety evaluation framework built on this idea.

### 1.2 PEGASUS 6-layer model (scenario *content*; Scholtes et al., 2021)

A separate concept: it structures **what a scenario contains**, not its level of abstraction.

| Layer | Content |
|---|---|
| L1 | Road network and road geometry |
| L2 | Traffic infrastructure (signs, barriers, markings) |
| L3 | Temporary modifications of L1 and L2 (road works) |
| L4 | Dynamic objects (vehicles, pedestrians, their manoeuvres) |
| L5 | Environment (weather, light) |
| L6 | Digital information (V2X, digital maps) |

PEGASUS (2016–2019) was a German research project, funded by the Federal Ministry for Economic Affairs and Energy, on methods for the safety assessment of highly automated driving.

In [ ]:
@dataclass
class ConcreteScenario:
    """One value per parameter: directly executable."""
    name: str
    road_type: str
    weather: str
    lighting: str
    pedestrian_speed: float      # m/s
    pedestrian_distance: float   # m
    ego_speed: float             # km/h
    occlusion: bool

    def describe(self):
        return (f"{self.name}: {self.road_type}, {self.weather}/{self.lighting}, pedestrian "
                f"{self.pedestrian_speed:.2f} m/s at {self.pedestrian_distance:.1f} m, ego {self.ego_speed:.1f} km/h, "
                f"occluded: {'yes' if self.occlusion else 'no'}")

@dataclass
class LogicalScenario:
    """Logical scenario: parameter ranges (bounded by the ODD)."""
    name: str
    road_type: List[str]
    weather_conditions: List[str]
    lighting_conditions: List[str]
    pedestrian_speed_range: Tuple[float, float]
    pedestrian_distance_range: Tuple[float, float]
    ego_speed_range: Tuple[float, float]

    def sample(self, n, rng):
        """Draw n concrete scenarios uniformly from the parameter space."""
        return [ConcreteScenario(
            name=f"{self.name}_{i + 1}",
            road_type=str(rng.choice(self.road_type)),
            weather=str(rng.choice(self.weather_conditions)),
            lighting=str(rng.choice(self.lighting_conditions)),
            pedestrian_speed=rng.uniform(*self.pedestrian_speed_range),
            pedestrian_distance=rng.uniform(*self.pedestrian_distance_range),
            ego_speed=rng.uniform(*self.ego_speed_range),
            occlusion=bool(rng.random() < 0.5)) for i in range(n)]

pedestrian_crossing = LogicalScenario(
    name="Pedestrian_Crossing",
    road_type=["urban_2lane", "residential"],
    weather_conditions=["clear", "rain", "fog"],
    lighting_conditions=["day", "night", "dusk"],
    pedestrian_speed_range=(0.5, 2.0),
    pedestrian_distance_range=(5.0, 30.0),
    ego_speed_range=(20.0, 50.0),
)
for s in pedestrian_crossing.sample(5, rng):
    print(s.describe())

### 1.3 Exploring the logical scenario

A simple **criticality proxy** helps to prioritise: here the time the ego vehicle needs to reach the crossing point, compared with the stopping time. The scoring below is **illustrative**; physically grounded criticality metrics (TTC, required deceleration, RSS) are covered in [04_SOTIF/05 Criticality metrics and RSS](../04_SOTIF_ISO21448/05_criticality_metrics_and_rss.ipynb).

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements.

In [ ]:
scen = pedestrian_crossing.sample(500, rng)
df = pd.DataFrame([s.__dict__ for s in scen])
v = df.ego_speed / 3.6                                         # m/s
ttc = df.pedestrian_distance / v                               # time to reach crossing point (s)
a_req = v ** 2 / (2 * df.pedestrian_distance)                  # deceleration needed to stop in time (m/s^2)
df["ttc_s"], df["a_req"] = ttc, a_req
df["critical"] = (a_req > 6.0) | ((ttc < 1.5) & df.occlusion)  # illustrative criticality rule

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ax = axes[0]
sc = ax.scatter(df.ego_speed, df.pedestrian_distance, c=df.a_req.clip(0, 10), cmap="viridis", s=18)
ax.scatter(df.ego_speed[df.critical], df.pedestrian_distance[df.critical], facecolors="none", edgecolors=OI[5], s=40,
           label="critical (illustrative rule)")
plt.colorbar(sc, ax=ax, label="Required deceleration (m/s², clipped at 10)")
ax.set_xlabel("Ego speed (km/h)"); ax.set_ylabel("Pedestrian distance (m)"); ax.set_title("Logical scenario: sampled concrete scenarios")
ax.legend()
ax = axes[1]
ax.hist(df.a_req, bins=40, color=OI[4], edgecolor="black")
ax.axvline(6.0, color=OI[5], ls="--", label="6 m/s² (illustrative limit)")
ax.set_xlabel("Required deceleration (m/s²)"); ax.set_ylabel("Number of scenarios"); ax.legend()
ax.set_title("Distribution of required deceleration")
ax = axes[2]
pd.crosstab(df.weather, df.lighting).plot.bar(ax=ax, color=[OI[0], OI[1], OI[7]], edgecolor="black", rot=0)
ax.set_xlabel("Weather"); ax.set_ylabel("Number of scenarios"); ax.set_title("Condition combinations sampled")
plt.tight_layout(); plt.show()

print(f"{len(df)} concrete scenarios; {df.critical.sum()} ({df.critical.mean():.1%}) flagged critical by the illustrative rule")
print(f"Required deceleration: median {df.a_req.median():.2f} m/s², 95th percentile {df.a_req.quantile(0.95):.2f} m/s²")
print("Smallest combination count:", pd.crosstab(df.weather, df.lighting).min().min(),
      "| largest:", pd.crosstab(df.weather, df.lighting).max().max())

Uniform sampling spends most of the budget on uncritical scenarios. Practical strategies therefore **search** the logical scenario space for critical regions (optimisation, importance sampling, adaptive sampling) and weight results back to the real-world exposure. Feng et al. (2023) showed with dense reinforcement learning how naturalistic and adversarial sampling can be combined for accelerated evaluation.

## 2. Simulation-based validation

### 2.1 Why simulation?
**Advantages:** safe, repeatable, scalable, controllable, full ground truth, rare events on demand.
**Challenges:** the sim-to-real gap (sensor models, physics, actor behaviour), validation of the simulator itself (tool confidence in ISO 26262-8 and ISO/PAS 8800), and modelling of unknown corner cases.

### 2.2 Common tools (as of October 2026)
- **CARLA** (open source): Unreal-Engine-based driving simulator with camera, LiDAR and radar sensor models and a Python API; version 0.10.0 (December 2024) moved to Unreal Engine 5.5 and added a native ROS 2 interface ([release note](https://carla.org/2024/12/19/release-0.10.0/)).
- **SUMO** (Eclipse / DLR, open source): **microscopic traffic-flow simulation** (traffic demand, routing, signal control, V2X via co-simulation, e.g. with CARLA). It renders no sensor data ([eclipse.dev/sumo](https://eclipse.dev/sumo/)).
- **esmini** and other **ASAM OpenSCENARIO** players: execution of standardised scenario descriptions.
- **Commercial:** e.g. IPG CarMaker, dSPACE, rFpro, Ansys, Applied Intuition, Foretellix (scenario-based V&V tooling).

The author's own work uses CARLA to evaluate LiDAR 3D object detectors in a SOTIF-related use case (Patel & Jung, VEHITS 2024).

### 2.3 Parameter-space coverage
We discretise the logical scenario into bins (3 weather × 3 lighting × 4 pedestrian-speed × 6 distance × 4 ego-speed × 2 occlusion = 1,728 bins). The bin index of a continuous parameter must be computed with edges that match the declared number of bins; otherwise some bins are unreachable and coverage can never reach 100 %.

In [ ]:
class CoverageAnalyzer:
    """Parameter-space (bin) coverage of a set of concrete scenarios."""
    def __init__(self, logical: LogicalScenario, n_speed=4, n_dist=6, n_ego=4):
        self.L = logical
        self.edges = {
            "pedestrian_speed": np.linspace(*logical.pedestrian_speed_range, n_speed + 1)[1:-1],
            "pedestrian_distance": np.linspace(*logical.pedestrian_distance_range, n_dist + 1)[1:-1],
            "ego_speed": np.linspace(*logical.ego_speed_range, n_ego + 1)[1:-1],
        }
        self.total_bins = (len(logical.weather_conditions) * len(logical.lighting_conditions)
                           * n_speed * n_dist * n_ego * 2)

    def bin_id(self, s: ConcreteScenario):
        return (s.weather, s.lighting,
                int(np.digitize(s.pedestrian_speed, self.edges["pedestrian_speed"])),
                int(np.digitize(s.pedestrian_distance, self.edges["pedestrian_distance"])),
                int(np.digitize(s.ego_speed, self.edges["ego_speed"])),
                s.occlusion)

    def coverage(self, scenarios):
        covered = {self.bin_id(s) for s in scenarios}
        return len(covered) / self.total_bins, len(covered)

analyzer = CoverageAnalyzer(pedestrian_crossing)
print(f"Coverage space: {analyzer.total_bins:,} bins")

N_MAX = 10_000
pool = pedestrian_crossing.sample(N_MAX, np.random.default_rng(7))     # generate once, then take prefixes
ns = np.unique(np.r_[np.arange(250, 2000, 250), np.arange(2000, N_MAX + 1, 1000)])
cov = np.array([analyzer.coverage(pool[:n])[0] for n in ns])
theory = 1 - (1 - 1 / analyzer.total_bins) ** ns                       # every bin equally likely here

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(ns, 100 * cov, "o-", color=OI[4], lw=2, label="Observed (uniform random sampling)")
ax1.plot(ns, 100 * theory, "k--", label="Expected: 1 - (1 - 1/B)^n")
for target, c in [(80, OI[2]), (95, OI[0])]:
    ax1.axhline(target, color=c, ls=":", label=f"{target} % coverage")
ax1.set_xlabel("Number of concrete scenarios"); ax1.set_ylabel("Bin coverage (%)"); ax1.set_title("Coverage growth"); ax1.legend()
ax2.bar(ns[1:], 100 * np.diff(cov) / np.diff(ns) * 1000, width=np.diff(ns) * 0.8, color=OI[1], edgecolor="black", align="edge")
ax2.set_xlabel("Number of concrete scenarios"); ax2.set_ylabel("Coverage gain per 1,000 extra scenarios (%)")
ax2.set_title("Diminishing returns")
plt.tight_layout(); plt.show()

for target in (0.80, 0.90, 0.95, 0.99):
    hit = ns[cov >= target]
    n_theory = np.log(1 - target) / np.log(1 - 1 / analyzer.total_bins)
    print(f"{target:.0%} coverage: observed at n = {hit[0]:,}" if len(hit) else f"{target:.0%} coverage: not reached with {N_MAX:,}",
          f"| expected n = {n_theory:,.0f}")

Uniform random sampling needs roughly $B\ln(1/(1-c))$ scenarios to cover a fraction $c$ of $B$ equally likely bins (the coupon-collector effect): about 5,200 for 95 % of 1,728 bins. A designed test (one scenario per bin, or a covering array) needs far fewer.

**Caution:** combinatorial bin coverage measures *input-space exploration*. It is not evidence that the residual risk in ISO 21448 Areas 2 and 3 is acceptable: one passing scenario per bin says little about failure probabilities inside the bin, and the bins themselves only cover the parameters someone thought of.

## 3. X-in-the-loop testing

| Stage | What is real | Typical use |
|---|---|---|
| **MIL / SIL** (model / software in the loop) | Nothing physical; simulated sensors, software under test | Algorithm development, large scenario sweeps, regression tests |
| **HIL** (hardware in the loop) | Target ECU / compute; simulated or stimulated sensors and vehicle | Timing, interfaces, integration, fault injection |
| **VIL** (vehicle in the loop, proving ground) | Real vehicle and sensors; soft targets (pedestrian dummies) | Dangerous scenarios under control, system validation |
| **Field operation** (FOT, shadow mode, deployment) | Everything | Long-tail discovery, statistical evidence, SPIs |

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements.

In [ ]:
levels = pd.DataFrame([
    ("SIL (simulation)",        1_000_000, 1,      "low-medium"),
    ("HIL (hardware in loop)",     10_000, 100,    "medium"),
    ("VIL (proving ground)",        1_000, 1_000,  "high"),
    ("Field operation",               100, 10_000, "real world"),
], columns=["level", "scenarios", "relative cost per scenario", "fidelity"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
cols = [OI[2], OI[1], OI[0], OI[5]]
ax1.barh(levels.level, levels.scenarios, color=cols, edgecolor="black"); ax1.set_xscale("log")
ax1.set_xlabel("Number of test scenarios (log scale, illustrative)"); ax1.invert_yaxis(); ax1.set_title("Volume")
ax2.barh(levels.level, levels["relative cost per scenario"], color=cols, edgecolor="black"); ax2.set_xscale("log")
ax2.set_xlabel("Relative cost per scenario (SIL = 1, log scale, illustrative)"); ax2.invert_yaxis(); ax2.set_title("Cost")
plt.tight_layout(); plt.show()
print(levels.to_string(index=False))

## 4. Statistical validation

### 4.1 Zero-failure demonstration
If failures occur as a Poisson process with rate $\lambda$ per mile and $n$ miles are driven **without failure**, then $\lambda < \lambda_0$ is shown with confidence $C$ when $e^{-\lambda_0 n} \le 1 - C$, i.e.
$$n \ge \frac{-\ln(1-C)}{\lambda_0}.$$
With $k$ observed failures, the one-sided upper bound is $\lambda_U = \chi^2_{C;\,2(k+1)}/(2n)$ (exact, Garwood). The functions below give the same results as `avsafety.validation` in this repository (see repo `avsafety/`).

### 4.2 Precision and superiority (RAND equations)
- **Precision:** to estimate the rate within $\pm\delta$ (relative) at confidence $1-\alpha$, about $x = (z_{1-\alpha/2}/\delta)^2$ failures must be observed, i.e. $n = x/\lambda$ miles (normal approximation).
- **Superiority with power:** to show that the AV rate $\lambda_1 = (1-A)\lambda_H$ is lower than the human rate $\lambda_H$ with significance $\alpha$ (one-sided) and power $1-\beta$: $n = \lambda_1\big((z_{1-\alpha} + z_{1-\beta})/(\lambda_H - \lambda_1)\big)^2$.

In [ ]:
def miles_to_demonstrate(rate_per_mile, confidence=0.95, failures=0):
    """Exposure such that `failures` observed events show rate < rate_per_mile (one-sided Poisson/chi-square bound)."""
    return stats.chi2.ppf(confidence, 2 * (failures + 1)) / (2 * rate_per_mile)

def poisson_rate_interval(events, exposure, confidence=0.95):
    """Exact (Garwood) two-sided confidence interval for a Poisson rate."""
    alpha = 1 - confidence
    lo = 0.0 if events == 0 else stats.chi2.ppf(alpha / 2, 2 * events) / 2
    hi = stats.chi2.ppf(1 - alpha / 2, 2 * (events + 1)) / 2
    return lo / exposure, hi / exposure

def fleet_years(miles, vehicles=100, miles_per_vehicle_year=24 * 365 * 25):
    """Years a fleet needs (RAND assumption: 24 h/day, 365 days/year, 25 mph)."""
    return miles / (vehicles * miles_per_vehicle_year)

def miles_precision(rate, delta=0.20, confidence=0.95):
    """RAND Eq. 4-5: miles to estimate the rate within +/- delta (relative)."""
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    return (z / delta) ** 2 / rate

def miles_superiority(rate_h, improvement=0.20, alpha=0.05, power=None):
    """RAND Eq. 7 (power=None) and Eq. 8 (with power): miles to show AV rate (1-A)*rate_h < rate_h."""
    lam1 = rate_h * (1 - improvement)
    z = stats.norm.ppf(1 - alpha) + (stats.norm.ppf(power) if power else 0.0)
    return lam1 * (z / (rate_h - lam1)) ** 2

H = 1.09e-8                     # US fatalities per mile, 2013 (as used by RAND)
rand_rows = [
    ("Zero failures: rate <= human, 95 % confidence", miles_to_demonstrate(H), "275 million", "12.5"),
    ("Estimate rate within +/-20 %, 95 % CI", miles_precision(H), "8.8 billion", "400"),
    ("Show 20 % better, 95 % confidence (no power req.)", miles_superiority(H, 0.20), "~5 billion", "225"),
    ("Show 20 % better, 95 % confidence, 80 % power", miles_superiority(H, 0.20, power=0.80), "11 billion", "518"),
    ("Show 5 % better, 95 % confidence, 80 % power", miles_superiority(H, 0.05, power=0.80), "215 billion", "~10,000"),
]
rand = pd.DataFrame([(q, m, fleet_years(m), fleet_years(m, vehicles=1), r_m, r_y) for q, m, r_m, r_y in rand_rows],
                    columns=["question", "miles (computed)", "years, 100 vehicles", "years, 1 vehicle",
                             "RAND miles", "RAND years (100 veh.)"])
fmt = rand.copy()
fmt["miles (computed)"] = fmt["miles (computed)"].map(lambda m: f"{m / 1e6:,.0f} M" if m < 1e9 else f"{m / 1e9:,.2f} bn")
fmt["years, 100 vehicles"] = fmt["years, 100 vehicles"].map(lambda y: f"{y:,.1f}")
fmt["years, 1 vehicle"] = fmt["years, 1 vehicle"].map(lambda y: f"{y:,.0f}")
print(fmt.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

ax = axes[0]
C = np.linspace(0.5, 0.995, 200)
for k, c in zip([0, 1, 5], [OI[4], OI[0], OI[5]]):
    ax.plot(100 * C, miles_to_demonstrate(H, C, k) / 1e6, color=c, lw=2, label=f"{k} failures observed")
ax.axvline(95, color="grey", ls=":")
ax.set_xlabel("Confidence C (%)"); ax.set_ylabel("Miles needed (millions)"); ax.set_yscale("log")
ax.set_title("Show fatality rate < 1.09 per 100 M miles"); ax.legend()

ax = axes[1]
Aimp = np.linspace(0.02, 0.9, 200)
ax.plot(100 * Aimp, miles_superiority(H, Aimp) / 1e9, color=OI[4], lw=2, label="95 % confidence")
ax.plot(100 * Aimp, miles_superiority(H, Aimp, power=0.8) / 1e9, color=OI[5], lw=2, label="95 % confidence, 80 % power")
ax.axvline(20, color="grey", ls=":")
ax.set_xlabel("True improvement over human fatality rate (%)"); ax.set_ylabel("Miles needed (billions)")
ax.set_yscale("log"); ax.set_title("Show AV is better than humans (RAND Eq. 7/8)"); ax.legend()

ax = axes[2]
fleets = np.array([1, 10, 100, 1000, 10000])
for (q, m, *_), c in zip(rand_rows[:4], OI):
    ax.plot(fleets, fleet_years(m, fleets), "o-", color=c, label=q.split(",")[0][:38])
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Fleet size (vehicles, 24/7 at 25 mph)"); ax.set_ylabel("Years of driving")
ax.set_title("Time to accumulate the miles"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

m0 = miles_to_demonstrate(H)
print(f"Zero-failure miles: {m0 / 1e6:.1f} M -> {fleet_years(m0):.1f} years with 100 vehicles, "
      f"{fleet_years(m0, 1):,.0f} years with one vehicle, {fleet_years(m0, 1000):.2f} years with 1,000 vehicles.")
print(f"One failure during the test raises the requirement to {miles_to_demonstrate(H, failures=1) / 1e6:.0f} M miles.")

**Context (as of October 2026):** the US fatality rate has been about 1.2–1.3 per 100 million vehicle miles in recent years (NHTSA reports 1.26 for 2023 and about 1.2 for 2024; [NHTSA FARS](https://www.nhtsa.gov/research-data/fatality-analysis-reporting-system-fars)). The conclusions do not change: fatality-level claims from driving alone remain out of reach before deployment, so arguments rely on surrogate events (injury and police-reported crashes, safety-relevant conflicts), scenario-based evidence and post-deployment monitoring.

### 4.3 Exact Poisson confidence intervals for observed rates

Suppose a fleet reports $k$ events of a given type in $n$ miles. The exact (Garwood) interval is based on the chi-square distribution. Its coverage is at least the nominal level for every true rate, at the price of being conservative. The **rule of three** is the special case $k=0$: the one-sided 95 % upper bound is $\approx 3/n$.

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements.

In [ ]:
exposure = 20e6                           # miles driven by a hypothetical fleet
benchmark = 2.0e-6                        # hypothetical human benchmark rate for the same event type (per mile)
rows = []
for k in [0, 1, 3, 10, 30, 100]:
    lo, hi = poisson_rate_interval(k, exposure)
    rows.append((k, k / exposure * 1e6, lo * 1e6, hi * 1e6,
                 "below benchmark" if hi < benchmark else ("above benchmark" if lo > benchmark else "inconclusive")))
ci = pd.DataFrame(rows, columns=["events k", "rate per M miles", "95 % CI low", "95 % CI high", "vs benchmark 2.0 per M"])
print(f"Exposure {exposure / 1e6:.0f} M miles")
print(ci.round(3).to_string(index=False))

# Monte Carlo check of the coverage of the exact interval
lam_true = 3e-7                           # true rate per mile (expected 6 events in 20 M miles)
k_sim = rng.poisson(lam_true * exposure, size=20_000)
lo_hi = np.array([poisson_rate_interval(k, exposure) for k in range(k_sim.max() + 1)])
covered = (lo_hi[k_sim, 0] <= lam_true) & (lam_true <= lo_hi[k_sim, 1])
print(f"\nSimulated coverage of the exact 95 % interval at a true rate of {lam_true * 1e6:.1f} per M miles: "
      f"{covered.mean():.3%} (nominal 95 %; exact intervals are conservative)")
print(f"Rule of three: zero events in {exposure / 1e6:.0f} M miles -> 95 % upper bound "
      f"{stats.chi2.ppf(0.95, 2) / 2 / exposure * 1e6:.3f} per M miles (approx. 3/n = {3 / exposure * 1e6:.3f})")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.errorbar(ci["events k"].astype(str), ci["rate per M miles"],
            yerr=[ci["rate per M miles"] - ci["95 % CI low"], ci["95 % CI high"] - ci["rate per M miles"]],
            fmt="o", color=OI[4], capsize=5, label="Observed rate with exact 95 % CI")
ax.axhline(benchmark * 1e6, color=OI[5], ls="--", label="Hypothetical benchmark")
ax.set_xlabel("Number of events observed in 20 M miles"); ax.set_ylabel("Events per million miles"); ax.set_yscale("symlog", linthresh=0.05)
ax.set_title("Few events give wide intervals"); ax.legend()
plt.tight_layout(); plt.show()

## 5. Validation metrics for AV perception

### 5.1 Standard ML metrics
- Object detection: precision, recall, average precision (AP/mAP) at IoU thresholds, nuScenes Detection Score.
- Classification: accuracy, confusion matrix, ROC/AUROC.

### 5.2 Safety-oriented metrics
1. **Misses that matter:** recall of vulnerable road users *within the required detection range and time*, evaluated at **object or track level**, not per frame.
2. **False positives that matter:** phantom objects that cause unnecessary emergency braking can lead to rear-end collisions; ISO 21448 treats such unintended behaviour as potentially hazardous too.
3. **Per-ODD-slice performance:** night, rain, occlusion, children, wheelchairs; worst slice rather than average.
4. **Uncertainty quality:** calibration ([04](04_calibration.ipynb)), OOD-monitor detection and false-alarm rates.
5. **Temporal consistency and latency:** track stability, no flickering detections, 99th-percentile latency.

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements.

In [ ]:
models = pd.DataFrame([
    ("Model A: balanced",        950,  50,  50, 950),
    ("Model B: high precision",  900,  10, 100, 990),
    ("Model C: high recall",     980, 200,  20, 800),
], columns=["model", "TP", "FP", "FN", "TN"]).set_index("model")
m = models
m["accuracy"] = (m.TP + m.TN) / m[["TP", "FP", "FN", "TN"]].sum(axis=1)
m["precision"] = m.TP / (m.TP + m.FP)
m["recall"] = m.TP / (m.TP + m.FN)
m["FNR"] = m.FN / (m.TP + m.FN)
m["FPR"] = m.FP / (m.FP + m.TN)
# Upper 95 % bound on the miss probability from n positives (exact binomial, Clopper-Pearson)
m["FNR 95 % upper"] = [stats.beta.ppf(0.95, fn + 1, tp) for tp, fn in zip(m.TP, m.FN)]

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
x = np.arange(len(m)); w = 0.25
ax = axes[0]
for i, (col, c) in enumerate([("accuracy", OI[4]), ("precision", OI[0]), ("recall", OI[2])]):
    ax.bar(x + (i - 1) * w, m[col], w, label=col, color=c, edgecolor="black")
ax.set_xticks(x); ax.set_xticklabels([s.split(":")[0] for s in m.index]); ax.set_ylim(0.7, 1.0)
ax.set_ylabel("Score"); ax.set_title("Standard metrics"); ax.legend()
ax = axes[1]
ax.bar(x - w / 2, m.FNR, w, label="FNR (missed objects)", color=OI[5], edgecolor="black")
ax.bar(x + w / 2, m.FPR, w, label="FPR (phantom objects)", color=OI[0], edgecolor="black")
ax.axhline(0.05, color="black", ls="--", label="Illustrative target 5 %")
ax.set_xticks(x); ax.set_xticklabels([s.split(":")[0] for s in m.index])
ax.set_ylabel("Error rate"); ax.set_title("Both error types can be hazardous"); ax.legend()
ax = axes[2]
cm = models.loc["Model B: high precision", ["TN", "FP", "FN", "TP"]].to_numpy().reshape(2, 2)
ax.imshow(cm, cmap="Blues"); ax.grid(False)
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=16, fontweight="bold",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred. negative", "Pred. positive"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["True negative", "True positive"])
ax.set_title(f"Model B confusion matrix (FNR {m.loc['Model B: high precision', 'FNR']:.1%})")
plt.tight_layout(); plt.show()

print(m.round(3).to_string())
b = m.loc["Model B: high precision"]
best_acc = m.accuracy.idxmax()
print(f"\nModel B: accuracy {b.accuracy:.1%} but FNR {b.FNR:.1%} (misses 1 in {1 / b.FNR:.0f} positives).")
print(f"Highest accuracy: {best_acc}; lowest FNR: {m.FNR.idxmin()}; lowest FPR: {m.FPR.idxmin()}.")
print("No model is best on all criteria: thresholds must come from a risk analysis that weighs missed objects "
      "against unintended braking, per ODD slice and at object/track level.")

## 6. Safety performance indicators and shadow mode

**Safety performance indicators (SPIs)**, central to UL 4600 (3rd ed., 2023), are metrics that provide evidence, during development and operation, that a safety-case claim holds, for example: rate of perception misses detected by a redundant channel per 1,000 km, rate of OOD-monitor triggers per ODD slice, rate of hard-braking events without a confirmed obstacle, time-to-detect for pedestrians in logged encounters. Each SPI has a threshold and a defined response when it is violated. ISO/TS 5083:2025 likewise covers post-deployment activities for automated driving systems, and ISO/PAS 8800 calls for measures during operation for AI systems ([01](01_ai_safety_standards.ipynb)).

**Shadow mode** runs a new perception or driving function passively alongside the active one in the fleet. Its outputs are logged and compared with the active system (and with what the driver did), without controlling the vehicle. It provides large amounts of real-world exposure for a new model version at no additional risk, and it finds disagreements that point to triggering conditions. Its limits: it cannot observe the *consequences* of the shadow function's actions (closed-loop effects), and disagreements still need labelling.

## 7. A validation plan template

The plan below is **illustrative** and shows how acceptance criteria can be phrased so that they are statistically meaningful. It illustrates V&V evidence that would feed into an ISO 21448 / ISO/PAS 8800 safety argument; it is not compliance evidence by itself.

In [ ]:
def zero_failure_upper_bound(n, confidence=0.95):
    """One-sided upper bound on a failure probability after n failure-free independent trials (exact binomial)."""
    return 1 - (1 - confidence) ** (1 / n)

def trials_for_bound(p_max, confidence=0.95):
    return int(np.ceil(np.log(1 - confidence) / np.log(1 - p_max)))

plan = [
    ("1. SIL (simulation)", "1,000,000 concrete scenarios from logical scenarios of the ODD",
     ["Recall >= 0.99 per ODD slice (lower 95 % bound), object level, within required range",
      "Unintended emergency-braking rate below project target per slice",
      "Calibration ECE (95 % CI upper bound) <= 0.05 per slice"]),
    ("2. HIL", "10,000 critical scenarios selected from SIL results",
     ["No regression versus SIL beyond statistical noise", "Latency p99 within budget"]),
    ("3. VIL (proving ground)", "n failure-free critical VRU scenarios",
     [f"0 misses in n = 1,000 trials -> miss probability <= {zero_failure_upper_bound(1000):.2%} (95 % conf.)",
      f"To show <= 0.1 % would need n = {trials_for_bound(0.001):,} failure-free trials"]),
    ("4. Field operation / shadow mode", "SPIs monitored per 1,000 km",
     ["SPI thresholds defined in advance, with response actions",
      "Surrogate event rates with exact Poisson CIs vs benchmark"]),
]
print("VALIDATION PLAN (illustrative): camera-based pedestrian detection, urban ODD <= 50 km/h\n")
for phase, scope, criteria in plan:
    print(f"{phase}\n   Scope: {scope}")
    for c in criteria:
        print(f"   - {c}")
    print()
print("Statistical note: zero failures in n independent trials bounds the failure probability by "
      f"1 - 0.05^(1/n) ~ 3/n at 95 % confidence (n = 1,000 -> {zero_failure_upper_bound(1000):.4f}).")
print("This template illustrates V&V evidence for an ISO 21448 / ISO/PAS 8800 safety argument; it is not compliance evidence by itself.")

## Key takeaways

- RAND (2016): **275 M failure-free miles** (≈12.5 years with 100 vehicles) to show "no worse than humans"; **8.8 bn miles** (≈400 years) to estimate the rate within ±20 %; **11 bn miles** (≈518 years) to show **20 % better with 95 % confidence and 80 % power**.
- Keep **abstraction levels** (functional → logical → concrete) and the **PEGASUS 6-layer content model** apart; the ODD bounds the logical scenarios.
- **SUMO** is a traffic-flow simulator, not a sensor simulator; CARLA provides sensor simulation.
- Bin coverage must be computed with consistent bin edges; it measures exploration, **not residual risk**.
- Use **exact Poisson (Garwood) intervals** and the **rule of three** for rare events; few events give wide intervals.
- Both **missed objects and phantom objects** can be hazardous; evaluate at object/track level, per ODD slice, with confidence bounds.
- Operation-phase evidence (**SPIs, shadow mode, field monitoring**) completes the argument (UL 4600, ISO/TS 5083, ISO/PAS 8800).

## Self-assessment

1. What is the difference between offline validation and runtime monitoring for AI safety?
2. How many failure-free miles show, with 95 % confidence, a fatality rate no higher than 1.09 per 100 M miles, and why does showing "20 % better" need far more?
3. What is a safety performance indicator (SPI) for an AI perception system? Give an example.
4. What can and cannot be learned from shadow-mode deployment?
5. Why is scenario-based testing preferred over purely random testing, and why is bin coverage not enough?
6. A VIL campaign shows 0 missed pedestrians in 300 trials. What can you claim at 95 % confidence?

<details>
<summary>Answers</summary>

1. Offline validation produces evidence before release on defined test data and scenarios; runtime monitoring checks, during operation, whether the assumptions behind that evidence still hold (ODD, input distribution, SPIs) and triggers responses. Both are needed.
2. About 275 million miles ($-\ln 0.05/1.09\times10^{-8}$). Showing superiority requires observing failures and statistically separating two close rates; with 95 % confidence and 80 % power for a 20 % improvement this needs about 11 billion miles.
3. A metric with a threshold that provides evidence for a safety claim, monitored during development and operation, e.g. the rate of pedestrian misses detected by a redundant LiDAR channel per 1,000 km, or the rate of OOD-monitor triggers per ODD slice.
4. Can: large real-world exposure for a new version without added risk; disagreements with the active system reveal candidate triggering conditions. Cannot: observe closed-loop consequences of the shadow system's actions; disagreements still need labelling and analysis.
5. Random driving rarely meets critical situations; scenarios target known critical regions and make tests reproducible. Bin coverage only measures exploration of the parameters someone chose; it does not bound failure probabilities inside bins or address unknown scenarios.
6. Miss probability ≤ 1 − 0.05^(1/300) ≈ 1.0 % (rule of three: 3/300), assuming independent, representative trials.
</details>

## References

- Kalra, N., Paddock, S. M. (2016). Driving to Safety: How Many Miles of Driving Would It Take to Demonstrate Autonomous Vehicle Reliability? RAND RR-1478. https://www.rand.org/pubs/research_reports/RR1478.html ; journal version: *Transportation Research Part A* 94, 182–193. https://doi.org/10.1016/j.tra.2016.09.010
- Menzel, T., Bagschik, G., Maurer, M. (2018). Scenarios for development, test and validation of automated vehicles. IEEE IV 2018. arXiv:1801.08598
- Scholtes, M. et al. (2021). 6-Layer model for a structured description and categorization of urban traffic and environment. *IEEE Access* 9, 59131–59147. https://doi.org/10.1109/ACCESS.2021.3072739
- Feng, S. et al. (2023). Dense reinforcement learning for safety validation of autonomous vehicles. *Nature* 615, 620–627. https://doi.org/10.1038/s41586-023-05732-2
- Garwood, F. (1936). Fiducial limits for the Poisson distribution. *Biometrika* 28(3/4), 437–442. https://doi.org/10.1093/biomet/28.3-4.437
- ISO 21448:2022; ISO/PAS 8800:2024; ISO 34502:2022 *Road vehicles — Test scenarios for automated driving systems — Scenario based safety evaluation framework*; ISO 34503:2023 *ODD taxonomy*; ISO/TS 5083:2025; UL 4600 (3rd ed., 2023).
- CARLA 0.10.0 release note (2024). https://carla.org/2024/12/19/release-0.10.0/ ; Eclipse SUMO. https://eclipse.dev/sumo/
- Patel, M., Jung, R. (2024). Simulation-Based Performance Evaluation of 3D Object Detection Methods with Deep Learning for a LiDAR Point Cloud Dataset in a SOTIF-related Use Case. VEHITS 2024. https://doi.org/10.5220/0012707300003702

---
**Navigation:** ← [06 Explainability](06_explainability.ipynb) · [Module index](README.md) · [Module 07: Runtime Assurance and Operations](../07_Runtime_Assurance_and_Operations/README.md) →

### Further practice

**Try this:** replace uniform sampling in Section 1 by importance sampling towards high required deceleration and re-weight the estimated failure rate; compute how many failure-free proving-ground trials are needed for a miss probability of 0.01 % at 99 % confidence.

The author's companion repository [AI-Perception-Safety-Workshop](https://github.com/milinpatel07/AI-Perception-Safety-Workshop) has further exercises.